In [ ]:
from matplotlib import pyplot as plt
import h5py
import numpy as np
import bacco

%matplotlib inline
%load_ext autoreload
%autoreload 2

In [ ]:
mtng = bacco.utils.load_MTNG(adr="/cosmos_storage/simulations/TNG_Family/MTNG/", snap=264)

In [ ]:
with h5py.File("/cosmos_storage/simulations/TNG_Family/MN5_resims/fiducial/hydro_output/snapdir_264/snapshot_264.2.hdf5") as f:
    print(f['Header'].attrs['MassTable'])
    print(f['Header'].attrs.keys())
    print(f['PartType3']['Masses'])

In [ ]:
sigma8 = 0.8159 #CHECK ME
ns     = 0.9667 #CHECK ME
tau    = 0.0965 #CHECK ME

snap = 264
zoom = {}

base = "/cosmos_storage/simulations/TNG_Family/MN5_resims/fiducial/hydro_output/"
zoom = bacco.Simulation(basedir=base, halo_file="groups_{:03d}/fof_subhalo_tab_{:03d}".format(snap,snap), sim_format='TNG500', fixedPk=True, use_orphans=False,\
                        tau=tau, ns=ns, sigma8=sigma8, dm_file="snapdir_{:03d}/snapshot_{:03d}".format(snap,snap), use_ids=True, numpart=4320)


In [ ]:
pos = np.vstack((zoom.gas['pos'], zoom.dm['pos'], zoom.lowres_dm['pos'], zoom.stars['pos'], zoom.bh['pos']))

dm_mass = np.ones(zoom.dm['pos'].shape[0])*zoom.header['ParticleMass']
mass = np.hstack((zoom.gas['mass'], dm_mass, zoom.lowres_dm['mass'], zoom.stars['mass'], zoom.bh['mass']))

In [ ]:
fig, ax = plt.subplots(dpi=100)
ax.hist(np.log10(mass))


In [ ]:
pk = bacco.statistics.compute_powerspectrum(
    ngrid=768,
    box=500,
    kmin=2*np.pi/500,
    kmax=np.pi/(500/768),
    nbins=30,
    pos=pos,
    log_binning=True,
    mass=mass,
    correct_grid=True,
    interlacing=True,
    cosmology=zoom.Cosmology,
    deconvolve_grid=True
    )

In [ ]:
kvec = np.logspace(-2,1,100)
pk_nlin = zoom.Cosmology.get_nlpowerspec_z(kvec, 1, cold=False)
pk_nlin_cold = zoom.Cosmology.get_nlpowerspec_z(kvec, 1, cold=True)

In [ ]:
fig, ax = plt.subplots(dpi=150, figsize=(5,4))

ax.set_xscale("log")
ax.set_yscale("log")

mask = np.where(pk['pk']!=0)

ax.plot(pk['k'][mask], pk['pk'][mask], color="C0", label="Multi-Zooms Power Spectrum")
ax.plot(kvec, pk_nlin, label="Non-Linear Power Spectrum", color='C3')

ax.set_xlabel(r"$k$ [$h$ Mpc$^{-1}$]")
ax.set_ylabel(r"$P(k)$ [Mpc$^3$ $h^{-3}$]")

ax.set_ylim(1e1, 1e5)

ax.legend(fontsize=8)

In [ ]:
fig, ax = plt.subplots(dpi=150, figsize=(5,4))

ax.set_xscale("log")

ax.axhline(1, color="k", ls="--", lw=2)

mask = np.where(pk['pk']!=0)
ax.plot(pk['k'][mask], pk['pk'][mask] / np.interp(pk['k'][mask], kvec, pk_nlin), color="C0", label="Multi-Zooms Power Spectrum")

ax.set_xlabel(r"$k$ [$h$ Mpc$^{-1}$]")
ax.set_xlim(0.1,5)
ax.set_ylim(0,1.2)

ax.legend(fontsize=8)

In [ ]:
fig, ax = plt.subplots(dpi=150, figsize=(5,4))

ax.set_xscale("log")

ax.axhline(0, color="k", ls="--", lw=2)

mask = np.where(pk['pk']!=0)
ax.plot(pk['k'][mask], np.log(pk['pk'][mask] / np.interp(pk['k'][mask], kvec, pk_nlin)), color="C0", label="Multi-Zooms Power Spectrum")

ax.set_xlabel(r"$k$ [$h$ Mpc$^{-1}$]")
ax.set_xlim(0.01,5)
ax.set_ylim(-0.5,0.5)

ax.legend(fontsize=8)